# SupportPilot: fine-tune a ticket router (week 11)

This notebook teaches a **small** model (Qwen2.5 1.5B) to classify CloudNotes support tickets
as well as a bigger prompted model, using LoRA with [Unsloth](https://github.com/unslothai/unsloth).
It runs on the **free Colab T4 GPU** in about 10-15 minutes.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. On your machine, generate the data: `cd backend && uv run python -m app.finetune.generate --count 500 --verify`
3. Upload `finetune/data/train.jsonl` and `finetune/data/test.jsonl` with the folder icon on the left.

**What you will learn:** LoRA (training a few million extra weights instead of all 1.5 billion),
4-bit loading (QLoRA), chat templates, training only on the answer, measuring before vs after,
and exporting to GGUF so Ollama can run the model locally.

In [ ]:
%%capture
!pip install unsloth

## 1. Load the base model in 4-bit

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"   # try "unsloth/Qwen2.5-0.5B-Instruct" for an even smaller router
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,       # QLoRA: base weights in 4-bit, so it fits a free GPU
)

## 2. Load the dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "train.jsonl", "test": "test.jsonl"})
print(ds)
print(ds["train"][0]["messages"])

## 3. Baseline: how does the base model do *before* training?

Same short prompt the fine-tuned model will get. Expect poor or badly formatted answers;
that is the point of fine-tuning.

In [ ]:
import json, re, torch

FIELDS = ("category", "priority", "sentiment")

def predict(messages):
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=48, do_sample=False)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    try:
        return json.loads(match.group(0)) if match else {}
    except json.JSONDecodeError:
        return {}

def evaluate(rows, limit=None):
    FastLanguageModel.for_inference(model)
    rows = rows.select(range(min(limit or len(rows), len(rows))))
    scores = {f: 0 for f in FIELDS}; exact = 0
    for row in rows:
        pred = predict(row["messages"])
        ok = {f: pred.get(f) == row["label"][f] for f in FIELDS}
        for f in FIELDS: scores[f] += ok[f]
        exact += all(ok.values())
    n = len(rows)
    result = {f"{f}_acc": round(scores[f] / n, 3) for f in FIELDS}
    result["exact_match"] = round(exact / n, 3)
    return result

before = evaluate(ds["test"])
before

## 4. Add LoRA adapters

Only these small adapter matrices are trained. `r` is their size: bigger learns more but
can overfit a 500-example dataset.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

## 5. Format with the chat template and train

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

def to_text(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

train = ds["train"].map(to_text)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=5,
        optim="adamw_8bit",
        output_dir="outputs",
        report_to="none",
        seed=42,
    ),
)
# Only learn to write the answer, not to repeat the ticket (Qwen's chat markers).
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()

## 6. After training: compare with the baseline

In [ ]:
after = evaluate(ds["test"])
print("before:", before)
print("after: ", after)

## 7. Export to GGUF for Ollama

`q4_k_m` is a 4-bit quantisation: about 1 GB, fast on a laptop CPU. The download can take
a few minutes.

In [ ]:
model.save_pretrained_gguf("supportpilot-router", tokenizer, quantization_method="q4_k_m")

In [ ]:
import glob
from google.colab import files

gguf = sorted(glob.glob("supportpilot-router*/*.gguf") + glob.glob("*.gguf"))
print(gguf)
files.download(gguf[-1])

## 8. Run it locally

Put the downloaded `.gguf` file in `finetune/` next to `Modelfile`, rename it to
`supportpilot-router.gguf`, then follow **Use the model** in `finetune/README.md`.

**Interview talking points:** why fine-tune instead of prompting (cost, latency, a shorter
prompt, consistent output); why LoRA/QLoRA (fits a free GPU, small adapters); why the
hand-written eval set matters more than the synthetic test split; when *not* to fine-tune
(facts that change belong in RAG, not in the weights).